<a href="https://colab.research.google.com/github/VictorPontual/LTSM_Attention_FRtoPT_Tradution/blob/main/FRtoPT_LTSM_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Carregar o dataset de tradução (francês -> português)
data_set = "ted_hrlr_translate/fr_to_pt"
train_data, val_data, test_data = tfds.load(data_set, split=["train", "validation", "test"], as_supervised=True)

# Definir os tokenizers para francês e português
tokenizer_en = tf.keras.layers.TextVectorization(
    standardize="lower_and_strip_punctuation", output_mode="int", output_sequence_length=40
)
tokenizer_pt = tf.keras.layers.TextVectorization(
    standardize="lower_and_strip_punctuation", output_mode="int", output_sequence_length=40
)

# Adaptar os tokenizers com o conjunto de dados
train_data_en = train_data.map(lambda en, pt: en).batch(32)
train_data_pt = train_data.map(lambda en, pt: pt).batch(32)

tokenizer_en.adapt(train_data_en)
tokenizer_pt.adapt(train_data_pt)

# Preparar os dados corretamente
train_input, decoder_input, train_target = prepare_data(train_data)
val_input, decoder_input_val, val_target = prepare_data(val_data)

# Criar lotes para os dados de treinamento
train_dataset = tf.data.Dataset.from_tensor_slices(((train_input, decoder_input), train_target))
train_dataset = train_dataset.batch(32).shuffle(buffer_size=1000)

# Criar lotes para os dados de validação
val_dataset = tf.data.Dataset.from_tensor_slices(((val_input, decoder_input_val), val_target))
val_dataset = val_dataset.batch(32)  # Não precisa de shuffle para dados de validação

# Construir e treinar o modelo
model = build_model(vocab_size_en, vocab_size_pt)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Treinamento com validação
model.fit(train_dataset, epochs=40, validation_data=val_dataset)

# Avaliação final
model.evaluate(val_dataset)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Preparar os dados corretamente
def prepare_data(data):
    input_seq = []
    target_seq = []
    for en, pt in tfds.as_numpy(data):
        input_seq.append(tokenizer_en([en])[0].numpy())
        target_seq.append(tokenizer_pt([pt])[0].numpy())

    input_tensor = tf.convert_to_tensor(input_seq, dtype=tf.int32)
    target_tensor = tf.convert_to_tensor(target_seq, dtype=tf.int32)

    decoder_input_tensor = tf.concat([tf.zeros_like(target_tensor[:, :1]), target_tensor[:, :-1]], axis=1)
    return input_tensor, decoder_input_tensor, target_tensor

In [ ]:
# Definir a camada de Atenção
class Attention(tf.keras.layers.Layer):
    def __init__(self):
        super(Attention, self).__init__()
        self.softmax = tf.keras.layers.Softmax(axis=-1)

    def call(self, query, value):
        attention_scores = tf.matmul(query, value, transpose_b=True)
        attention_weights = self.softmax(attention_scores)
        context_vector = tf.matmul(attention_weights, value)
        return context_vector

# Definir o modelo
def build_model(vocab_size_en, vocab_size_pt, embedding_dim=128, lstm_units=128):
    # Entradas
    encoder_input = tf.keras.layers.Input(shape=(None,))
    decoder_input = tf.keras.layers.Input(shape=(None,))

    # Embedding
    encoder_embedding = tf.keras.layers.Embedding(vocab_size_en, embedding_dim)(encoder_input)
    decoder_embedding = tf.keras.layers.Embedding(vocab_size_pt, embedding_dim)(decoder_input)

    # LSTM para o codificador (encoder)
    encoder_lstm = tf.keras.layers.LSTM(lstm_units, return_sequences=True, return_state=True, dropout=0.2)
    encoder_output, state_h, state_c = encoder_lstm(encoder_embedding)

    # LSTM para o decodificador (decoder)
    decoder_lstm = tf.keras.layers.LSTM(lstm_units, return_sequences=True, return_state=False, dropout=0.2)
    decoder_output = decoder_lstm(decoder_embedding, initial_state=[state_h, state_c])

    # Aplicar atenção
    attention_layer = Attention()
    attention = attention_layer(decoder_output, encoder_output)

    # Concatenar atenção e saída do decodificador
    context_and_decoder = tf.keras.layers.Concatenate(axis=-1)([decoder_output, attention])

    # Camada final
    output = tf.keras.layers.Dense(vocab_size_pt, activation='softmax')(context_and_decoder)

    model = tf.keras.Model(inputs=[encoder_input, decoder_input], outputs=output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Tamanho do vocabulário
vocab_size_en = len(tokenizer_en.get_vocabulary())
vocab_size_pt = len(tokenizer_pt.get_vocabulary())

# Construir o modelo
model = build_model(vocab_size_en, vocab_size_pt)

# Resumo do modelo
model.summary()

# Preparar dados de treinamento
train_input, decoder_input, train_target = prepare_data(train_data)

# Criar lotes para os dados de treinamento
train_dataset = tf.data.Dataset.from_tensor_slices(((train_input, decoder_input), train_target))
train_dataset = train_dataset.batch(32).shuffle(buffer_size=1000)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4             │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_5             │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_4 (Embedding)   │ (None, None, 128)      │      4,715,264 │ input_layer_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_5 (Embedding)   │ (None, None, 128)      │      4,786,560 │ input_layer_5[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_4 (LSTM)             │ [(None, None, 128),    │        131,584 │ embedding_4[0][0]      │
│                           │ (None, 128), (None,    │                │                        │
│                           │ 128)]                  │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_5 (LSTM)             │ (None, None, 128)      │        131,584 │ embedding_5[0][0],     │
│                           │                        │                │ lstm_4[0][1],          │
│                           │                        │                │ lstm_4[0][2]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attention (Attention)     │ (None, None, 128)      │              0 │ lstm_5[0][0],          │
│                           │                        │                │ lstm_4[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, None, 256)      │              0 │ lstm_5[0][0],          │
│                           │                        │                │ attention[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, None, 37395)    │      9,610,515 │ concatenate[0][0]      │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 19,375,507 (73.91 MB)

 Trainable params: 19,375,507 (73.91 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Treinar o modelo
model.fit(train_dataset, epochs=40)

Epoch 1/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 155s 109ms/step - accuracy: 0.6492 - loss: 3.1316
Epoch 2/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 149s 109ms/step - accuracy: 0.6817 - loss: 2.2177
Epoch 3/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 149s 109ms/step - accuracy: 0.7083 - loss: 1.9413
Epoch 4/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 149s 109ms/step - accuracy: 0.7386 - loss: 1.6697
Epoch 5/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 149s 109ms/step - accuracy: 0.7634 - loss: 1.4392
Epoch 6/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 202s 109ms/step - accuracy: 0.7853 - loss: 1.2416
Epoch 7/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 149s 109ms/step - accuracy: 0.8026 - loss: 1.0855
Epoch 8/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 149s 109ms/step - accuracy: 0.8198 - loss: 0.9586
Epoch 9/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 202s 109ms/step - accuracy: 0.8387 - loss: 0.8406
Epoch 10/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 149s 109ms/step - accuracy: 0.8534 - loss: 0.7499
Epoch 11/40
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 202s 109ms/step - accuracy: 0.8641 -

In [ ]:
# Preparar dados de validação
val_input, decoder_input, val_target = prepare_data(val_data)

# Criar lotes para os dados de validação
val_dataset = tf.data.Dataset.from_tensor_slices(((val_input, decoder_input), val_target))
val_dataset = val_dataset.batch(32)  # Não há necessidade de shuffle para dados de validação


# Avaliar o modelo
model.evaluate(val_dataset)

36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.7522 - loss: 2.5630


[2.5764071941375732, 0.7535146474838257]

In [ ]:
def translate(text):
    # Obter vocabulário
    vocab_pt = tokenizer_pt.get_vocabulary()
    vocab_en = tokenizer_en.get_vocabulary()

    # Verificar índices de <start> e <end>
    start_token = vocab_pt.index('<start>') if '<start>' in vocab_pt else 0
    end_token = vocab_pt.index('<end>') if '<end>' in vocab_pt else 1

    # Tokenizar a entrada
    tokenized_input = tokenizer_en(tf.constant([text]))  # Adicionar batch dimension
    tokenized_input = tf.cast(tokenized_input, tf.int32)

    # Inicializar o decodificador com "<start>"
    decoder_input = tf.constant([[start_token]])

    translated_tokens = []
    for _ in range(40):  # Limite de 40 tokens
        # Fazer a previsão
        predictions = model.predict([tokenized_input, decoder_input], verbose=0)
        predicted_token = tf.argmax(predictions[:, -1, :], axis=-1).numpy()[0]

        # Adicionar token à tradução
        if predicted_token == end_token:
            break
        translated_tokens.append(predicted_token)

        # Atualizar a entrada do decodificador
        decoder_input = tf.concat([decoder_input, tf.constant([[predicted_token]])], axis=-1)

    # Converter tokens traduzidos para texto
    translated_text = " ".join([vocab_pt[token] for token in translated_tokens if token < len(vocab_pt)])

    return translated_text

# Exemplo de tradução
print(translate("comment ça s'appelle"))

como é que se chama                                   
